# 01 — Clean Occupancy Data


In [1]:
import pandas as pd

occupancy = pd.read_csv("../data/raw/ladot_occupancy/20261017T052111914317Z/Sensor Transactions_2025_08.csv")

occupancy.head()

,SpaceID,EventTime_Local,EventTime_UTC,OccupancyState
0,WV308,8/1/2025 12:00:01 AM,8/1/2025 7:00:01 AM,VACANT
1,WV65,8/1/2025 12:00:01 AM,8/1/2025 7:00:01 AM,VACANT
2,SV24,8/1/2025 12:00:01 AM,8/1/2025 7:00:01 AM,OCCUPIED
3,WV582,8/1/2025 12:00:02 AM,8/1/2025 7:00:02 AM,OCCUPIED
4,WV126,8/1/2025 12:00:02 AM,8/1/2025 7:00:02 AM,VACANT


In [2]:
print("Rows and columns:", occupancy.shape)

print("\nColumn types:")
print(occupancy.dtypes)

print("\nUnique parking spaces:")
print(occupancy["SpaceID"].nunique())

Rows and columns: (3707820, 4)

Column types:
SpaceID            object
EventTime_Local    object
EventTime_UTC      object
OccupancyState     object
dtype: object

Unique parking spaces:
4905


In [3]:
occupancy.isnull().sum()

SpaceID            0
EventTime_Local    0
EventTime_UTC      0
OccupancyState     0
dtype: int64

In [4]:

print("Occupancy states:")
print(occupancy["OccupancyState"].value_counts())


print("\nExact duplicate rows:")
print(occupancy.duplicated().sum())


occupancy["EventTime_Local"] = pd.to_datetime(occupancy["EventTime_Local"])

print("\nTime range:")
print(occupancy["EventTime_Local"].min())
print(occupancy["EventTime_Local"].max())

Occupancy states:
OccupancyState
VACANT      1849424
OCCUPIED    1847148
UNKNOWN       11248
Name: count, dtype: int64

Exact duplicate rows:
0


/var/folders/h_/6xwp29sx51bb6tfjw793j8z80000gn/T/ipykernel_79489/218040131.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  occupancy["EventTime_Local"] = pd.to_datetime(occupancy["EventTime_Local"])



Time range:
2025-08-01 00:00:01
2025-08-31 23:59:59


In [5]:
space_counts = occupancy.groupby("SpaceID").size()

space_counts.describe()

count    4905.000000
mean      755.926606
std       672.564195
min         1.000000
25%       389.000000
50%       589.000000
75%       878.000000
max      8517.000000
dtype: float64

In [6]:
coverage = occupancy.groupby("SpaceID")["EventTime_Local"].agg(["min", "max", "count"])

coverage["days_covered"] = (coverage["max"] - coverage["min"]).dt.days

coverage["days_covered"].describe()

count    4905.000000
mean       29.046687
std         2.000729
min         0.000000
25%        29.000000
50%        29.000000
75%        30.000000
max        30.000000
Name: days_covered, dtype: float64

In [7]:
occupancy["date"] = occupancy["EventTime_Local"].dt.date

active_days = occupancy.groupby("SpaceID")["date"].nunique()

active_days.describe()

count    4905.000000
mean       26.622222
std         4.177373
min         1.000000
25%        26.000000
50%        26.000000
75%        31.000000
max        31.000000
Name: date, dtype: float64

In [8]:
example = occupancy[occupancy["SpaceID"] == "WV308"]

example.head(15)

,SpaceID,EventTime_Local,EventTime_UTC,OccupancyState,date
0,WV308,2025-08-01 00:00:01,8/1/2025 7:00:01 AM,VACANT,2025-08-01
10546,WV308,2025-08-01 06:11:05,8/1/2025 1:11:05 PM,OCCUPIED,2025-08-01
11425,WV308,2025-08-01 06:26:35,8/1/2025 1:26:35 PM,VACANT,2025-08-01
11490,WV308,2025-08-01 06:27:47,8/1/2025 1:27:47 PM,OCCUPIED,2025-08-01
11599,WV308,2025-08-01 06:29:23,8/1/2025 1:29:23 PM,VACANT,2025-08-01
14330,WV308,2025-08-01 07:03:38,8/1/2025 2:03:38 PM,OCCUPIED,2025-08-01
15280,WV308,2025-08-01 07:12:47,8/1/2025 2:12:47 PM,VACANT,2025-08-01
15311,WV308,2025-08-01 07:13:12,8/1/2025 2:13:12 PM,OCCUPIED,2025-08-01
16890,WV308,2025-08-01 07:28:03,8/1/2025 2:28:03 PM,VACANT,2025-08-01
17694,WV308,2025-08-01 07:35:15,8/1/2025 2:35:15 PM,OCCUPIED,2025-08-01


In [9]:
occupancy = occupancy.sort_values(["SpaceID", "EventTime_Local"])

occupancy["next_time"] = occupancy.groupby("SpaceID")["EventTime_Local"].shift(-1)

occupancy["duration_minutes"] = (
    occupancy["next_time"] - occupancy["EventTime_Local"]
).dt.total_seconds() / 60

occupancy[["SpaceID", "EventTime_Local", "OccupancyState", "next_time", "duration_minutes"]].head(15)

,SpaceID,EventTime_Local,OccupancyState,next_time,duration_minutes
18964,61111,2025-08-01 07:46:06,VACANT,2025-08-01 07:57:00,10.9
20302,61111,2025-08-01 07:57:00,OCCUPIED,2025-08-01 08:01:42,4.7
20957,61111,2025-08-01 08:01:42,VACANT,2025-08-01 08:09:12,7.5
22006,61111,2025-08-01 08:09:12,OCCUPIED,2025-08-01 08:09:18,0.1
22018,61111,2025-08-01 08:09:18,VACANT,2025-08-01 09:00:30,51.2
28963,61111,2025-08-01 09:00:30,OCCUPIED,2025-08-01 09:01:54,1.4
29224,61111,2025-08-01 09:01:54,VACANT,2025-08-01 10:07:00,65.1
40391,61111,2025-08-01 10:07:00,OCCUPIED,2025-08-01 10:13:42,6.7
41443,61111,2025-08-01 10:13:42,VACANT,2025-08-01 10:22:18,8.6
42712,61111,2025-08-01 10:22:18,OCCUPIED,2025-08-01 10:42:06,19.8


In [10]:
occupancy["duration_minutes"].describe(percentiles=[0.5, 0.9, 0.95, 0.99])

count    3.702915e+06
mean     5.648012e+01
std      2.368703e+02
min      1.666667e-02
50%      4.316667e+00
90%      1.044167e+02
95%      2.928167e+02
99%      8.915310e+02
max      3.259072e+04
Name: duration_minutes, dtype: float64

In [11]:
print("Over 12 hours:", (occupancy["duration_minutes"] > 720).sum())
print("Over 24 hours:", (occupancy["duration_minutes"] > 1440).sum())
print("Over 48 hours:", (occupancy["duration_minutes"] > 2880).sum())
print("Over 72 hours:", (occupancy["duration_minutes"] > 4320).sum())

Over 12 hours: 59740
Over 24 hours: 14994
Over 48 hours: 1734
Over 72 hours: 711


In [12]:
occupancy["valid_duration"] = (
    occupancy["duration_minutes"].notna()
    & (occupancy["duration_minutes"] <= 1440)
    & (occupancy["OccupancyState"] != "UNKNOWN")
)

occupancy["valid_duration"].value_counts()

valid_duration
True     3677140
False      30680
Name: count, dtype: int64

In [13]:
valid = occupancy[occupancy["valid_duration"]].copy()

valid["occupied_minutes"] = valid["duration_minutes"].where(
    valid["OccupancyState"] == "OCCUPIED", 0
)

space_occupancy = valid.groupby("SpaceID").agg(
    occupied_minutes=("occupied_minutes", "sum"),
    total_minutes=("duration_minutes", "sum")
)

space_occupancy["occupancy_rate"] = (
    space_occupancy["occupied_minutes"] / space_occupancy["total_minutes"]
)

space_occupancy["occupancy_rate"].describe()

count    4901.000000
mean        0.553006
std         0.208118
min         0.000000
25%         0.410367
50%         0.554946
75%         0.706238
max         1.000000
Name: occupancy_rate, dtype: float64

In [14]:
space_occupancy["active_days"] = active_days

space_occupancy = space_occupancy[
    space_occupancy["active_days"] >= 25
].copy()

print("Final number of spaces:", len(space_occupancy))

space_occupancy["occupancy_rate"].describe()

Final number of spaces: 4185


count    4185.000000
mean        0.535638
std         0.196711
min         0.003520
25%         0.399502
50%         0.536993
75%         0.678971
max         0.998916
Name: occupancy_rate, dtype: float64

In [15]:
space_occupancy = space_occupancy.reset_index()

space_occupancy.to_csv(
    "../data/processed/space_occupancy_aug2025.csv",
    index=False
)

space_occupancy.head()

,SpaceID,occupied_minutes,total_minutes,occupancy_rate,active_days
0,61111,19511.316667,37847.833333,0.515520,30
1,61112,17954.033333,38293.550000,0.468853,30
2,61113,21939.050000,40604.650000,0.540309,30
3,61114,26049.100000,43845.700000,0.594108,31
4,61116,22993.916667,43823.916667,0.524689,31
